In [1]:
from duckdb_assistant import DuckDBAssistant

In [2]:
dda = DuckDBAssistant()

In [3]:
prompt = """
I have a website https://raw.githubusercontent.com/duckdb/duckdb-web/refs/heads/main/docs/current/guides/sql_features/graph_queries.md
which I want to extract text from and then load to a DuckDB table.
"""

In [4]:
resp = dda.generate(prompt)

In [5]:
resp

{'query': "CREATE TABLE graph_queries_doc AS\nSELECT *\nFROM read_text('https://raw.githubusercontent.com/duckdb/duckdb-web/refs/heads/main/docs/current/guides/sql_features/graph_queries.md');",
 'response_text': "```sql\nCREATE TABLE graph_queries_doc AS\nSELECT *\nFROM read_text('https://raw.githubusercontent.com/duckdb/duckdb-web/refs/heads/main/docs/current/guides/sql_features/graph_queries.md');\n```"}

In [6]:
dda.dd.execute(resp["query"])

In [8]:
prompt = """
Is there a DuckDB extension which writes a DuckDB table to Chroma?
"""

In [9]:
resp = dda.generate(prompt, explain_results = True)

In [10]:
resp["response_text"]

'Currently, **there is no official or native DuckDB extension** that allows writing a DuckDB table directly to ChromaDB using pure DuckDB SQL (e.g., via a `COPY ... TO` command or custom destination driver). \n\n---\n\n### Key Context & Alternatives\n\n1. **Chroma\'s Historical Relationship with DuckDB:**\n   In its earlier versions, Chroma actually used DuckDB under the hood as its default local persistence engine alongside `hnswlib`. However, Chroma has since evolved its backend architecture, and DuckDB does not currently expose a direct extension sink to push data into a live Chroma instance.\n\n2. **Native Vector Search inside DuckDB (`vss` Extension):**\n   If your goal is simply to perform vector similarity searches without needing Chroma specifically, DuckDB has an official **`vss` (Vector Similarity Search)** extension. It supports `FLOAT[N]` array types and HNSW indexing directly inside DuckDB:\n   ```sql\n   INSTALL vss;\n   LOAD vss;\n\n   -- Create an HNSW index on a vector

In [ ]:

import chroma_db
import duckdb

# 1. Connect to DuckDB and retrieve table data
con = duckdb.connect("my_database.duckdb")

# Fetch data as PyArrow Table or Python Dicts
results = con.execute(
    """
    SELECT 
        id::VARCHAR AS id,
        embedding_column AS embeddings,
        text_content AS documents,
        {\'category\': category} AS metadatas
    FROM my_vector_table
"""
).fetchall()

# Format for Chroma
ids = [row[0] for row in results]
embeddings = [row[1] for row in results]
documents = [row[2] for row in results]
metadatas = [row[3] for row in results]

# 2. Connect to Chroma and insert
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="duckdb_collection")

collection.add(
    ids=ids, embeddings=embeddings, documents=documents, metadatas=metadatas
)
